**ACGAN**

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from torch.nn.utils import spectral_norm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
IMG_SIZE = 112
CHANNELS = 3
LATENT_DIM = 512
NUM_CLASSES = 2
BATCH_SIZE = 64
EPOCHS = 2200
LR = 0.0001

os.makedirs("images", exist_ok=True)

In [3]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_dataset = datasets.ImageFolder("newdata/train", transform=transform)
dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

FileNotFoundError: [WinError 3] 系统找不到指定的路径。: 'newdata/train'

In [6]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()

        self.label_emb = nn.Embedding(NUM_CLASSES, NUM_CLASSES)

        self.init_size = IMG_SIZE // 16  # 7

        self.l1 = nn.Sequential(
            nn.Linear(LATENT_DIM + NUM_CLASSES, 1024 * self.init_size ** 2)
        )

        self.conv_blocks = nn.Sequential(
            nn.BatchNorm2d(1024),

            nn.ConvTranspose2d(1024, 512, 4, 2, 1),
            nn.BatchNorm2d(512),
            nn.ReLU(True),

            nn.ConvTranspose2d(512, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            nn.ConvTranspose2d(64, CHANNELS, 3, 1, 1),
            nn.Tanh()
        )

    def forward(self, z, labels):
        label_input = self.label_emb(labels)
        gen_input = torch.cat((z, label_input), dim=1)

        out = self.l1(gen_input)
        out = out.view(out.size(0), 1024, self.init_size, self.init_size)

        img = self.conv_blocks(out)
        return img

In [7]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        def block(in_feat, out_feat, stride=2):
            return [
                spectral_norm(nn.Conv2d(in_feat, out_feat, 3, stride, 1)),
                nn.LeakyReLU(0.2, inplace=True),
                nn.Dropout(0.4)
            ]

        self.model = nn.Sequential(
            *block(CHANNELS, 32, 1),
            *block(32, 64),
            *block(64, 128),
            *block(128, 256),
            *block(256, 512)
        )

        ds_size = IMG_SIZE // 16

        self.adv_layer = nn.Sequential(
            nn.Linear(512 * ds_size * ds_size, 1),
            nn.Sigmoid()
        )

        self.aux_layer = nn.Sequential(
            nn.Linear(512 * ds_size * ds_size, NUM_CLASSES),
            nn.Softmax(dim=1)
        )

    def forward(self, img):
        out = self.model(img)
        out = out.view(out.size(0), -1)

        validity = self.adv_layer(out)
        label = self.aux_layer(out)

        return validity, label

In [9]:
generator = Generator().to(device)
discriminator = Discriminator().to(device)

adversarial_loss = nn.BCELoss()
auxiliary_loss = nn.CrossEntropyLoss()

optimizer_G = optim.Adam(generator.parameters(), lr=LR, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=LR, betas=(0.5, 0.999))

In [10]:
def sample_images(epoch):
    generator.eval()

    z = torch.randn(5, LATENT_DIM).to(device)
    labels = torch.tensor([0,0,1,1,1]).to(device)

    gen_imgs = generator(z, labels)
    gen_imgs = gen_imgs * 0.5 + 0.5

    fig, axs = plt.subplots(1, 5, figsize=(12,3))

    for i in range(5):
        img = gen_imgs[i].cpu().permute(1,2,0).numpy()
        axs[i].imshow(img)
        axs[i].set_title(f"class {labels[i].item()}")
        axs[i].axis('off')

    plt.savefig(f"images/epoch_{epoch}.png")
    plt.close()

In [4]:
def generate_cxr(generator, n_per_class=50, save_dir="output"):
    generator.eval()

    covid_dir = os.path.join(save_dir, "CovidCXR")
    noncovid_dir = os.path.join(save_dir, "NonCovidCXR")

    os.makedirs(covid_dir, exist_ok=True)
    os.makedirs(noncovid_dir, exist_ok=True)

    with torch.no_grad():

        # COVID = 0
        for i in range(n_per_class):
            z = torch.randn(1, LATENT_DIM).to(device)
            label = torch.tensor([0]).to(device)

            img = generator(z, label)
            img = (img * 0.5 + 0.5).cpu().numpy()[0]
            img = (img * 255).astype(np.uint8)
            img = np.transpose(img, (1, 2, 0))

            Image.fromarray(img).save(f"{covid_dir}/{i+1}.jpeg")

        # NON-COVID = 1
        for i in range(n_per_class):
            z = torch.randn(1, LATENT_DIM).to(device)
            label = torch.tensor([1]).to(device)

            img = generator(z, label)
            img = (img * 0.5 + 0.5).cpu().numpy()[0]
            img = (img * 255).astype(np.uint8)
            img = np.transpose(img, (1, 2, 0))

            Image.fromarray(img).save(f"{noncovid_dir}/{i+1}.jpeg")

In [ ]:
g_losses = []
d_losses = []

for epoch in range(EPOCHS):
    for i, (imgs, labels) in enumerate(dataloader):

        batch_size = imgs.size(0)

        real = torch.ones(batch_size, 1).to(device)
        fake = torch.zeros(batch_size, 1).to(device)

        imgs = imgs.to(device)
        labels = labels.to(device)

        # ------------------
        # Train Generator
        # ------------------
        optimizer_G.zero_grad()

        z = torch.randn(batch_size, LATENT_DIM).to(device) * 0.8
        gen_labels = torch.randint(0, NUM_CLASSES, (batch_size,)).to(device)

        gen_imgs = generator(z, gen_labels)

        validity, pred_label = discriminator(gen_imgs)

        g_loss = adversarial_loss(validity, real) + 1.0 * auxiliary_loss(pred_label, gen_labels)
        g_loss.backward()
        optimizer_G.step()

        # ------------------
        # Train Discriminator
        # ------------------
        optimizer_D.zero_grad()

        real_pred, real_aux = discriminator(imgs)
        d_real_loss = adversarial_loss(real_pred, real) + auxiliary_loss(real_aux, labels)

        fake_pred, fake_aux = discriminator(gen_imgs.detach())
        d_fake_loss = adversarial_loss(fake_pred, fake) + auxiliary_loss(fake_aux, gen_labels)

        d_loss = (d_real_loss + d_fake_loss) / 2
        d_loss.backward()
        optimizer_D.step()

    g_losses.append(g_loss.item())
    d_losses.append(d_loss.item())

    print(f"[Epoch {epoch}] D loss: {d_loss.item():.4f}, G loss: {g_loss.item():.4f}")

    # ✅ 每50轮生成
    if epoch % 50 == 0:
        sample_images(epoch)

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(g_losses, label="Generator Loss")
plt.plot(d_losses, label="Discriminator Loss")
plt.legend()
plt.title("Training Loss")
plt.show()

In [8]:
generator = Generator()  # 你的模型类

generator.load_state_dict(
    torch.load("ACGANgenerator2.pth", map_location=device)
)

generator.to(device)
generator.eval()

C:\Users\12656\AppData\Local\Temp\ipykernel_88052\2881665130.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("ACGANgenerator2.pth", map_location=device)


Generator(
  (label_emb): Embedding(2, 2)
  (l1): Sequential(
    (0): Linear(in_features=514, out_features=50176, bias=True)
  )
  (conv_blocks): Sequential(
    (0): BatchNorm2d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (1): ConvTranspose2d(1024, 512, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (2): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ReLU(inplace=True)
    (4): ConvTranspose2d(512, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (5): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): ConvTranspose2d(256, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): ConvTranspose2d(128, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (11): BatchNorm2d(64, eps=1e-05, momentum=0.1, 

In [9]:
generate_cxr(generator, n_per_class=50, save_dir="output")